In [ ]:
import cv2
import os
import numpy as np

# 1) Loading my image , clicked in a garden!

# look for the image, imread reads the image ans stores in the img
img = cv2.imread("image.jpeg")

# if image does not exist, just crash with a clear message.
if img is None:
    raise FileNotFoundError("couldn't find image.png / image.jpg / image.jpeg in this folder")

# 2) convert to grayscale manually.

def to_grayscale(img):

    # OpenCV stores image as BGR, not RGB
    # So:
    # img[:,:,0] -> Blue
    # img[:,:,1] -> Green
    # img[:,:,2] -> Red

    b = img[:, :, 0]
    g = img[:, :, 1]
    r = img[:, :, 2]

    # Formula:
    # gray = 0.299R + 0.587G + 0.114B
    # This gives visually realistic brightness
    gray = (0.299 * r + 0.587 * g + 0.114 * b)

    # Convert float values back to uint8
    # because image pixels must be in range 0-255
    return gray.astype(np.uint8)


# Apply grayscale conversion
gray = to_grayscale(img)

# Save grayscale image into folder
cv2.imwrite("image_gray.png", gray)
print("Part 2 is Done!")

# 3) SALT AND PEPPER - 25% of pixels will be flipped to either black or white
total_pixels = gray.size
num_noisy = int(0.25 * total_pixels)

noisy = gray.copy()
rows = gray.shape[0]
cols = gray.shape[1]

# Making SALT pixels — turn some pixels white (greyscale value - 255)
salt_r = np.random.randint(0, rows, num_noisy // 2)
salt_c = np.random.randint(0, cols, num_noisy // 2)
noisy[salt_r, salt_c] = 255

# Making PEPPER pixels — turn some pixels black (greyscale value - 0)
pepper_r = np.random.randint(0, rows, num_noisy // 2)
pepper_c = np.random.randint(0, cols, num_noisy // 2)
noisy[pepper_r, pepper_c] = 0

# Save noisy image into folder
cv2.imwrite("image_noisy.png", noisy)
print("done part 3!")

# 4) using Adaptive median filter!
# This will Remove noise!

def median_filter(image, size=3):
    pad = size // 2                                       #adds padding in the image
    padded = np.pad(image, pad, mode='reflect')
    output = image.copy()

    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            window = padded[r:r+size, c:c+size].flatten()      # This will create a array of 9 numbers in 3x3 area, to find median
            output[r, c] = np.median(window)                   # Median value is now the output[r,c] value, therefore values like 0 or 255 dissapears.

    return output

denoised = median_filter(noisy)
cv2.imwrite("image_denoised.png", denoised)
print("done Part 4!")

# 5)  find edges — pixels that differ a lot from their neighbors are boundaries! or around whom, big contrast is present...

# sobel Filters — detect intensity changes in x and y direction
kx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]])

ky = np.array([[-1, -2, -1],
               [ 0,  0,  0],
               [ 1,  2,  1]])

padded = np.pad(denoised, 1, mode='reflect')
gx = np.zeros_like(denoised, dtype=float)
gy = np.zeros_like(denoised, dtype=float)

for r in range(denoised.shape[0]):
    for c in range(denoised.shape[1]):
        window = padded[r:r+3, c:c+3]
        gx[r, c] = np.sum(kx * window)
        gy[r, c] = np.sum(ky * window)

# combine x and y gradients to get Magnitude (edge strength)
edges = np.sqrt(gx**2 + gy**2)

# anything above threshold is a boundary
edges = (edges > 100).astype(np.uint8) * 255
#  This converts grayscale edge strength → binary image!
#  Makes True  - 1 , False - 0 and then
#  1 ->255 (white) and 0 ->0 (black)
#  So we get the Final Edges Image!

cv2.imwrite("image_boundary.png", edges)
print("done Part 5!")

# 6) Use Otsu’s thresholding technique for making the binary image of it.

# Convert the full image into one long 1D list for threshold finding
pixels = denoised.flatten()

# We will try every possible threshold from 0 to 255
# and keep track of the BEST
best_thresh = 0

# This stores the best "separation score" found till now
best_variance = 0
min_within_variance = float('inf')

# Try every threshold one by one
for t in range(256):

    below = pixels[pixels <= t]    # Blacks
    above = pixels[pixels > t]     # Whites

    # If one side becomes empty, then threshold is totally bad
    if len(below) == 0 or len(above) == 0:
        continue

    # ratio of each class
    w0 = len(below) / len(pixels)
    w1 = len(above) / len(pixels)

    # Mean of each class
    mean0 = np.mean(below)
    mean1 = np.mean(above)

    # Variance of each class
    var0 = np.var(below)
    var1 = np.var(above)

    #Final Main Variance
    within_variance = (w0 * var0) + (w1 * var1)


    # If this threshold gives better separation, then update...
    if within_variance < min_within_variance:
        min_within_variance = within_variance
        best_thresh = t

# Final threshold selected by Otsu
print(f"otsu threshold: {best_thresh}")

# Convert grayscale image into binary image  (black and white basically)

# Pixels greater than threshold -> white (255)
# Pixels less than or equal -> black (0)
binary = (denoised > best_thresh).astype(np.uint8) * 255


# Save final binary image
cv2.imwrite("image_binary.png", binary)
print("done last Part, Part 6!")



Part 2 is Done!
done part 3!
done Part 4!
done Part 5!
otsu threshold: 136
done last Part, Part 6!
